# DRD2-Hi — OOD Holdout vs Random Shuffle Protocol Comparison

This notebook compares two inner hyperparameter-selection protocols on the DRD2-Hi task:

1. **OOD holdout**: the validation subset is chemically separated from the inner training subset.
2. **Random shuffle**: the validation subset is obtained by randomly splitting the outer training set.

The main experimental question is:

> Does in-distribution hyperparameter selection produce higher internal validation scores without improving final out-of-distribution test performance?

The analysis focuses on:

- inner validation PR-AUC;
- inner train PR-AUC;
- outer train PR-AUC after refitting;
- final OOD test PR-AUC;
- inner-to-test generalization gap;
- train-to-test generalization gap;
- selected hyperparameters across folds.

The models considered are:

- Decision Tree;
- Logistic Regression;
- Linear SVM.

The fingerprints considered are:

- ECFP4;
- MACCS;
- RDKit descriptors, where available.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("../..").resolve()
RESULTS_ROOT = PROJECT_ROOT / "results" / "hi" / "drd2"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RESULTS_ROOT: {RESULTS_ROOT}")
print(f"RESULTS_ROOT exists: {RESULTS_ROOT.exists()}")

PROJECT_ROOT: /home/f.capria/drug-discovery-lohi
RESULTS_ROOT: /home/f.capria/drug-discovery-lohi/results/hi/drd2
RESULTS_ROOT exists: True


# Experiment registry

In [2]:
EXPERIMENTS = [
    # ---------------------------------------------------------------------
    # Decision Tree
    # ---------------------------------------------------------------------
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "dt_drd2_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "dt_drd2_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "dt_drd2_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "dt_drd2_hi_random_shuffle_maccs",
    },

    # ---------------------------------------------------------------------
    # Logistic Regression
    # ---------------------------------------------------------------------
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "lr_drd2_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "lr_drd2_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "OOD holdout",
        "result_dir": "lr_drd2_hi_inner_ood_holdout_rdkit_desc",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "lr_drd2_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "lr_drd2_hi_random_shuffle_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "Random shuffle",
        "result_dir": "lr_drd2_hi_random_shuffle_rdkit_desc",
    },

    # ---------------------------------------------------------------------
    # Linear SVM
    # ---------------------------------------------------------------------
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_drd2_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_drd2_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_drd2_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_drd2_hi_random_shuffle_maccs",
    },
]

registry = pd.DataFrame(EXPERIMENTS)
registry["path"] = registry["result_dir"].apply(lambda d: RESULTS_ROOT / d)
registry["exists"] = registry["path"].apply(lambda p: p.exists())

registry

,model,model_short,fingerprint,protocol,result_dir,path,exists
0,Decision Tree,DT,ECFP4,OOD holdout,dt_drd2_hi_inner_ood_holdout_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/drd2/dt_drd2_hi_inner_ood_holdout_ecfp4,True
1,Decision Tree,DT,MACCS,OOD holdout,dt_drd2_hi_inner_ood_holdout_maccs,/home/f.capria/drug-discovery-lohi/results/hi/drd2/dt_drd2_hi_inner_ood_holdout_maccs,True
2,Decision Tree,DT,ECFP4,Random shuffle,dt_drd2_hi_random_shuffle_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/drd2/dt_drd2_hi_random_shuffle_ecfp4,True
3,Decision Tree,DT,MACCS,Random shuffle,dt_drd2_hi_random_shuffle_maccs,/home/f.capria/drug-discovery-lohi/results/hi/drd2/dt_drd2_hi_random_shuffle_maccs,True
4,Logistic Regression,LR,ECFP4,OOD holdout,lr_drd2_hi_inner_ood_holdout_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/drd2/lr_drd2_hi_inner_ood_holdout_ecfp4,True
5,Logistic Regression,LR,MACCS,OOD holdout,lr_drd2_hi_inner_ood_holdout_maccs,/home/f.capria/drug-discovery-lohi/results/hi/drd2/lr_drd2_hi_inner_ood_holdout_maccs,True
6,Logistic Regression,LR,RDKit desc,OOD holdout,lr_drd2_hi_inner_ood_holdout_rdkit_desc,/home/f.capria/drug-discovery-lohi/results/hi/drd2/lr_drd2_hi_inner_ood_holdout_rdkit_desc,True
7,Logistic Regression,LR,ECFP4,Random shuffle,lr_drd2_hi_random_shuffle_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/drd2/lr_drd2_hi_random_shuffle_ecfp4,True
8,Logistic Regression,LR,MACCS,Random shuffle,lr_drd2_hi_random_shuffle_maccs,/home/f.capria/drug-discovery-lohi/results/hi/drd2/lr_drd2_hi_random_shuffle_maccs,True
9,Logistic Regression,LR,RDKit desc,Random shuffle,lr_drd2_hi_random_shuffle_rdkit_desc,/home/f.capria/drug-discovery-lohi/results/hi/drd2/lr_drd2_hi_random_shuffle_rdkit_desc,True


In [3]:
missing = registry.loc[~registry["exists"], ["model", "fingerprint", "protocol", "path"]]

if len(missing) > 0:
    print("Missing result folders:")
    display(missing)
else:
    print("All registered result folders exist.")

All registered result folders exist.


# Load params_fold_i.json into df_folds

In [4]:
def load_params_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def safe_get_metric(metrics: dict, key: str):
    if metrics is None:
        return np.nan
    return metrics.get(key, np.nan)


rows = []

for exp in EXPERIMENTS:
    result_path = RESULTS_ROOT / exp["result_dir"]

    for fold in [1, 2, 3]:
        params_path = result_path / f"params_fold_{fold}.json"

        if not params_path.exists():
            print(f"Missing file: {params_path}")
            continue

        data = load_params_json(params_path)

        train_metrics = data.get("train_metrics", {})
        test_metrics = data.get("test_metrics", {})
        best_params = data.get("best_params", {})

        inner_pr_auc = data.get("inner_selection_score", np.nan)
        inner_train_pr_auc = data.get("inner_train_score", np.nan)
        train_pr_auc = safe_get_metric(train_metrics, "pr_auc")
        test_pr_auc = safe_get_metric(test_metrics, "pr_auc")

        row = {
            "model": exp["model"],
            "model_short": exp["model_short"],
            "fingerprint": exp["fingerprint"],
            "protocol": exp["protocol"],
            "result_dir": exp["result_dir"],
            "fold": fold,

            "inner_pr_auc": inner_pr_auc,
            "inner_train_pr_auc": inner_train_pr_auc,
            "train_pr_auc": train_pr_auc,
            "test_pr_auc": test_pr_auc,

            "inner_test_gap": inner_pr_auc - test_pr_auc,
            "train_test_gap": train_pr_auc - test_pr_auc,

            "best_params": best_params,
            "inner_split_strategy": data.get("inner_split_strategy", None),
            "time_seconds": data.get("time_seconds", np.nan),
        }

        rows.append(row)

df_folds = pd.DataFrame(rows)

order_model = {
    "Decision Tree": 0,
    "Logistic Regression": 1,
    "Linear SVM": 2,
}

order_fp = {
    "ECFP4": 0,
    "MACCS": 1,
    "RDKit desc": 2,
}

order_protocol = {
    "OOD holdout": 0,
    "Random shuffle": 1,
}

df_folds["model_order"] = df_folds["model"].map(order_model)
df_folds["fingerprint_order"] = df_folds["fingerprint"].map(order_fp)
df_folds["protocol_order"] = df_folds["protocol"].map(order_protocol)

df_folds = df_folds.sort_values(
    ["model_order", "fingerprint_order", "protocol_order", "fold"]
).reset_index(drop=True)

print(f"Loaded rows: {len(df_folds)}")
df_folds.head(2)

Loaded rows: 42


,model,model_short,fingerprint,protocol,result_dir,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,best_params,inner_split_strategy,time_seconds,model_order,fingerprint_order,protocol_order
0,Decision Tree,DT,ECFP4,OOD holdout,dt_drd2_hi_inner_ood_holdout_ecfp4,1,0.855177,0.917171,0.9080,0.6979,0.157277,0.2101,"{'ccp_alpha': 0.001, 'class_weight': None, 'criterion': 'gini', 'max_depth': 20, 'max_features': 'sqrt', 'min_sample...",holdout,9.5,0,0,0
1,Decision Tree,DT,ECFP4,OOD holdout,dt_drd2_hi_inner_ood_holdout_ecfp4,2,0.699841,0.873181,0.8189,0.7698,-0.069959,0.0491,"{'ccp_alpha': 0.001, 'class_weight': None, 'criterion': 'gini', 'max_depth': 15, 'max_features': 'log2', 'min_sample...",holdout,6.2,0,0,0


# Per-fold table

In [5]:
per_fold_table = df_folds[
    [
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "inner_pr_auc",
        "inner_train_pr_auc",
        "train_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
    ]
].copy()

numeric_cols = [
    "inner_pr_auc",
    "inner_train_pr_auc",
    "train_pr_auc",                                                                                         
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

per_fold_table[numeric_cols] = per_fold_table[numeric_cols].round(4)

per_fold_table

,model,fingerprint,protocol,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Decision Tree,ECFP4,OOD holdout,1,0.8552,0.9172,0.9080,0.6979,0.1573,0.2101
1,Decision Tree,ECFP4,OOD holdout,2,0.6998,0.8732,0.8189,0.7698,-0.0700,0.0491
2,Decision Tree,ECFP4,OOD holdout,3,0.7375,0.9056,0.8232,0.6815,0.0560,0.1417
3,Decision Tree,ECFP4,Random shuffle,1,0.9115,0.9602,0.9522,0.6237,0.2878,0.3285
4,Decision Tree,ECFP4,Random shuffle,2,0.8444,0.9043,0.8757,0.7540,0.0904,0.1217
5,Decision Tree,ECFP4,Random shuffle,3,0.9097,0.9596,0.9573,0.6544,0.2553,0.3029
6,Decision Tree,MACCS,OOD holdout,1,0.8427,0.9366,0.9679,0.6262,0.2165,0.3417
7,Decision Tree,MACCS,OOD holdout,2,0.7410,0.8619,0.8447,0.7569,-0.0159,0.0878
8,Decision Tree,MACCS,OOD holdout,3,0.7214,0.8456,0.7870,0.6575,0.0639,0.1295
9,Decision Tree,MACCS,Random shuffle,1,0.9158,0.9563,0.9554,0.5913,0.3245,0.3641


# Aggregated summary table

In [6]:
def mean_std_string(mean_value, std_value, digits=4):
    if pd.isna(mean_value):
        return ""
    if pd.isna(std_value):
        return f"{mean_value:.{digits}f}"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"


summary_numeric = (
    df_folds
    .groupby(["model", "model_short", "fingerprint", "protocol"], as_index=False)
    .agg(
        inner_mean=("inner_pr_auc", "mean"),
        inner_std=("inner_pr_auc", "std"),
        inner_train_mean=("inner_train_pr_auc", "mean"),
        inner_train_std=("inner_train_pr_auc", "std"),
        train_mean=("train_pr_auc", "mean"),
        train_std=("train_pr_auc", "std"),
        test_mean=("test_pr_auc", "mean"),
        test_std=("test_pr_auc", "std"),
        inner_test_gap_mean=("inner_test_gap", "mean"),
        inner_test_gap_std=("inner_test_gap", "std"),
        train_test_gap_mean=("train_test_gap", "mean"),
        train_test_gap_std=("train_test_gap", "std"),
    )
)

summary_numeric["model_order"] = summary_numeric["model"].map(order_model)
summary_numeric["fingerprint_order"] = summary_numeric["fingerprint"].map(order_fp)
summary_numeric["protocol_order"] = summary_numeric["protocol"].map(order_protocol)

summary_numeric = summary_numeric.sort_values(
    ["model_order", "fingerprint_order", "protocol_order"]
).reset_index(drop=True)

summary_table = summary_numeric[
    ["model", "fingerprint", "protocol"]
].copy()

summary_table["Inner PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_mean"], r["inner_std"]), axis=1
)

summary_table["Inner train PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_train_mean"], r["inner_train_std"]), axis=1
)

summary_table["Train PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["train_mean"], r["train_std"]), axis=1
)

summary_table["Final OOD test PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["test_mean"], r["test_std"]), axis=1
)

summary_table["Inner-test gap"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_test_gap_mean"], r["inner_test_gap_std"]), axis=1
)

summary_table["Train-test gap"] = summary_numeric.apply(
    lambda r: mean_std_string(r["train_test_gap_mean"], r["train_test_gap_std"]), axis=1
)

summary_table

,model,fingerprint,protocol,Inner PR-AUC,Inner train PR-AUC,Train PR-AUC,Final OOD test PR-AUC,Inner-test gap,Train-test gap
0,Decision Tree,ECFP4,OOD holdout,0.7642 ± 0.0810,0.8987 ± 0.0228,0.8500 ± 0.0502,0.7164 ± 0.0470,0.0478 ± 0.1138,0.1336 ± 0.0808
1,Decision Tree,ECFP4,Random shuffle,0.8885 ± 0.0382,0.9414 ± 0.0321,0.9284 ± 0.0457,0.6774 ± 0.0681,0.2112 ± 0.1059,0.2510 ± 0.1127
2,Decision Tree,MACCS,OOD holdout,0.7684 ± 0.0651,0.8814 ± 0.0485,0.8665 ± 0.0924,0.6802 ± 0.0682,0.0882 ± 0.1181,0.1863 ± 0.1362
3,Decision Tree,MACCS,Random shuffle,0.8881 ± 0.0349,0.9522 ± 0.0043,0.9502 ± 0.0048,0.6739 ± 0.0826,0.2141 ± 0.1164,0.2763 ± 0.0856
4,Logistic Regression,ECFP4,OOD holdout,0.7514 ± 0.0991,0.9568 ± 0.0197,0.9441 ± 0.0347,0.7572 ± 0.0996,-0.0058 ± 0.1918,0.1869 ± 0.0985
5,Logistic Regression,ECFP4,Random shuffle,0.9155 ± 0.0306,0.9831 ± 0.0121,0.9803 ± 0.0117,0.7677 ± 0.1102,0.1479 ± 0.1369,0.2126 ± 0.1196
6,Logistic Regression,MACCS,OOD holdout,0.7377 ± 0.0592,0.9451 ± 0.0213,0.9080 ± 0.0143,0.7461 ± 0.0920,-0.0083 ± 0.1508,0.1619 ± 0.1032
7,Logistic Regression,MACCS,Random shuffle,0.8707 ± 0.0138,0.9186 ± 0.0201,0.9126 ± 0.0179,0.7516 ± 0.0799,0.1191 ± 0.0914,0.1609 ± 0.0963
8,Logistic Regression,RDKit desc,OOD holdout,0.7643 ± 0.0765,0.9300 ± 0.0146,0.9053 ± 0.0181,0.7883 ± 0.0809,-0.0240 ± 0.1453,0.1171 ± 0.0885
9,Logistic Regression,RDKit desc,Random shuffle,0.8775 ± 0.0343,0.9358 ± 0.0180,0.9302 ± 0.0193,0.7713 ± 0.0893,0.1062 ± 0.1204,0.1589 ± 0.1075


# Delta table

In [7]:
pivot_summary = summary_numeric.pivot_table(
    index=["model", "model_short", "fingerprint"],
    columns="protocol",
    values=[
        "inner_mean",
        "test_mean",
        "inner_test_gap_mean",
        "train_test_gap_mean",
    ],
)

delta_rows = []

for idx, row in pivot_summary.iterrows():
    model, model_short, fingerprint = idx

    try:
        random_inner = row[("inner_mean", "Random shuffle")]
        ood_inner = row[("inner_mean", "OOD holdout")]

        random_test = row[("test_mean", "Random shuffle")]
        ood_test = row[("test_mean", "OOD holdout")]

        random_inner_gap = row[("inner_test_gap_mean", "Random shuffle")]
        ood_inner_gap = row[("inner_test_gap_mean", "OOD holdout")]

        random_train_gap = row[("train_test_gap_mean", "Random shuffle")]
        ood_train_gap = row[("train_test_gap_mean", "OOD holdout")]

    except KeyError:
        continue

    delta_rows.append({
        "model": model,
        "model_short": model_short,
        "fingerprint": fingerprint,

        "ood_inner_mean": ood_inner,
        "random_inner_mean": random_inner,
        "delta_inner": random_inner - ood_inner,

        "ood_test_mean": ood_test,
        "random_test_mean": random_test,
        "delta_test": random_test - ood_test,

        "ood_inner_test_gap": ood_inner_gap,
        "random_inner_test_gap": random_inner_gap,
        "delta_inner_test_gap": random_inner_gap - ood_inner_gap,

        "ood_train_test_gap": ood_train_gap,
        "random_train_test_gap": random_train_gap,
        "delta_train_test_gap": random_train_gap - ood_train_gap,
    })

delta_table = pd.DataFrame(delta_rows)

delta_table["model_order"] = delta_table["model"].map(order_model)
delta_table["fingerprint_order"] = delta_table["fingerprint"].map(order_fp)

delta_table = delta_table.sort_values(
    ["model_order", "fingerprint_order"]
).reset_index(drop=True)

delta_display = delta_table[
    [
        "model",
        "fingerprint",
        "ood_inner_mean",
        "random_inner_mean",
        "delta_inner",
        "ood_test_mean",
        "random_test_mean",
        "delta_test",
        "ood_inner_test_gap",
        "random_inner_test_gap",
        "delta_inner_test_gap",
        "delta_train_test_gap",
    ]
].copy()

delta_numeric_cols = delta_display.select_dtypes(include=[np.number]).columns
delta_display[delta_numeric_cols] = delta_display[delta_numeric_cols].round(4)

delta_display

,model,fingerprint,ood_inner_mean,random_inner_mean,delta_inner,ood_test_mean,random_test_mean,delta_test,ood_inner_test_gap,random_inner_test_gap,delta_inner_test_gap,delta_train_test_gap
0,Decision Tree,ECFP4,0.7642,0.8885,0.1244,0.7164,0.6774,-0.0390,0.0478,0.2112,0.1634,0.1174
1,Decision Tree,MACCS,0.7684,0.8881,0.1197,0.6802,0.6739,-0.0063,0.0882,0.2141,0.1259,0.0900
2,Logistic Regression,ECFP4,0.7514,0.9155,0.1641,0.7572,0.7677,0.0105,-0.0058,0.1479,0.1536,0.0257
3,Logistic Regression,MACCS,0.7377,0.8707,0.1330,0.7461,0.7516,0.0056,-0.0083,0.1191,0.1274,-0.0010
4,Logistic Regression,RDKit desc,0.7643,0.8775,0.1132,0.7883,0.7713,-0.0170,-0.0240,0.1062,0.1302,0.0419
5,Linear SVM,ECFP4,0.7377,0.9093,0.1717,0.7532,0.7725,0.0192,-0.0156,0.1369,0.1524,0.0135
6,Linear SVM,MACCS,0.7459,0.8674,0.1215,0.7394,0.7410,0.0016,0.0065,0.1264,0.1199,0.0048


# Hyperparameter summary tables

In [8]:
def flatten_best_params(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in df.iterrows():
        base = {
            "model": row["model"],
            "fingerprint": row["fingerprint"],
            "protocol": row["protocol"],
            "fold": row["fold"],
            "inner_pr_auc": row["inner_pr_auc"],
            "test_pr_auc": row["test_pr_auc"],
        }

        params = row["best_params"]

        if isinstance(params, dict):
            for key, value in params.items():
                clean_key = key.replace("model__", "")
                base[clean_key] = value

        rows.append(base)

    return pd.DataFrame(rows)


hp_df = flatten_best_params(df_folds)

hp_df["model_order"] = hp_df["model"].map(order_model)
hp_df["fingerprint_order"] = hp_df["fingerprint"].map(order_fp)
hp_df["protocol_order"] = hp_df["protocol"].map(order_protocol)

hp_df = hp_df.sort_values(
    ["model_order", "fingerprint_order", "protocol_order", "fold"]
).reset_index(drop=True)

for col in ["inner_pr_auc", "test_pr_auc"]:
    hp_df[col] = hp_df[col].round(4)

hp_df.head()

,model,fingerprint,protocol,fold,inner_pr_auc,test_pr_auc,ccp_alpha,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,C,l1_ratio,model_order,fingerprint_order,protocol_order
0,Decision Tree,ECFP4,OOD holdout,1,0.8552,0.6979,0.0010,NaN,gini,20.0,sqrt,5.0,20.0,NaN,NaN,0,0,0
1,Decision Tree,ECFP4,OOD holdout,2,0.6998,0.7698,0.0010,NaN,gini,15.0,log2,5.0,20.0,NaN,NaN,0,0,0
2,Decision Tree,ECFP4,OOD holdout,3,0.7375,0.6815,0.0001,NaN,gini,7.0,log2,1.0,5.0,NaN,NaN,0,0,0
3,Decision Tree,ECFP4,Random shuffle,1,0.9115,0.6237,0.0000,balanced,gini,20.0,sqrt,5.0,2.0,NaN,NaN,0,0,1
4,Decision Tree,ECFP4,Random shuffle,2,0.8444,0.7540,0.0001,NaN,gini,15.0,sqrt,5.0,20.0,NaN,NaN,0,0,1


# Logistic Regression hyperparameters

In [9]:
lr_hp_table = hp_df.loc[
    hp_df["model"] == "Logistic Regression",
    [
        "protocol",
        "fingerprint",
        "fold",
        "C",
        "class_weight",
        "l1_ratio",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

lr_hp_table = lr_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

lr_hp_table

,protocol,fingerprint,fold,C,class_weight,l1_ratio,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,0.10,balanced,0.25,0.8607,0.6795
1,OOD holdout,ECFP4,2,0.10,balanced,0.00,0.6675,0.8695
2,OOD holdout,ECFP4,3,0.05,balanced,1.00,0.7261,0.7226
3,Random shuffle,ECFP4,1,1.00,balanced,0.25,0.9366,0.6486
4,Random shuffle,ECFP4,2,0.50,balanced,0.25,0.8805,0.8660
5,Random shuffle,ECFP4,3,0.10,balanced,0.00,0.9295,0.7884
6,OOD holdout,MACCS,1,1.00,NaN,1.00,0.8043,0.6403
7,OOD holdout,MACCS,2,10.00,NaN,1.00,0.6910,0.8075
8,OOD holdout,MACCS,3,10.00,balanced,1.00,0.7180,0.7904
9,Random shuffle,MACCS,1,10.00,balanced,0.50,0.8827,0.6598


# Svm liner hyperparameters

In [10]:
svm_hp_table = hp_df.loc[
    hp_df["model"] == "Linear SVM",
    [
        "protocol",
        "fingerprint",
        "fold",
        "C",
        "class_weight",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

svm_hp_table = svm_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

svm_hp_table

,protocol,fingerprint,fold,C,class_weight,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,100.000,NaN,0.8531,0.6434
1,OOD holdout,ECFP4,2,0.010,balanced,0.6738,0.8613
2,OOD holdout,ECFP4,3,0.001,NaN,0.6862,0.7550
3,Random shuffle,ECFP4,1,0.250,balanced,0.9339,0.6587
4,Random shuffle,ECFP4,2,0.100,balanced,0.8724,0.8714
5,Random shuffle,ECFP4,3,0.100,balanced,0.9217,0.7873
6,OOD holdout,MACCS,1,1.000,NaN,0.8099,0.6489
7,OOD holdout,MACCS,2,25.000,balanced,0.7131,0.7832
8,OOD holdout,MACCS,3,50.000,balanced,0.7147,0.7860
9,Random shuffle,MACCS,1,25.000,balanced,0.8789,0.6559


# Decision Tree hyperparameters

In [11]:
dt_hp_table = hp_df.loc[
    hp_df["model"] == "Decision Tree",
    [
        "protocol",
        "fingerprint",
        "fold",
        "criterion",
        "max_depth",
        "max_features",
        "min_samples_leaf",
        "min_samples_split",
        "ccp_alpha",
        "class_weight",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

dt_hp_table = dt_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

dt_hp_table

,protocol,fingerprint,fold,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,ccp_alpha,class_weight,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,gini,20.0,sqrt,5.0,20.0,0.0010,NaN,0.8552,0.6979
1,OOD holdout,ECFP4,2,gini,15.0,log2,5.0,20.0,0.0010,NaN,0.6998,0.7698
2,OOD holdout,ECFP4,3,gini,7.0,log2,1.0,5.0,0.0001,NaN,0.7375,0.6815
3,Random shuffle,ECFP4,1,gini,20.0,sqrt,5.0,2.0,0.0000,balanced,0.9115,0.6237
4,Random shuffle,ECFP4,2,gini,15.0,sqrt,5.0,20.0,0.0001,NaN,0.8444,0.7540
5,Random shuffle,ECFP4,3,entropy,20.0,NaN,10.0,2.0,0.0000,NaN,0.9097,0.6544
6,OOD holdout,MACCS,1,gini,15.0,log2,2.0,5.0,0.0001,NaN,0.8427,0.6262
7,OOD holdout,MACCS,2,gini,7.0,sqrt,5.0,2.0,0.0001,balanced,0.7410,0.7569
8,OOD holdout,MACCS,3,entropy,7.0,sqrt,1.0,5.0,0.0100,NaN,0.7214,0.6575
9,Random shuffle,MACCS,1,gini,20.0,sqrt,5.0,2.0,0.0000,balanced,0.9158,0.5913


# Compact hyperparameter set summary

In [12]:
def unique_values_as_string(series: pd.Series) -> str:
    values = []
    for value in series.dropna().tolist():
        if value not in values:
            values.append(value)
    return "{" + ", ".join(str(v) for v in values) + "}"


hp_set_summary_rows = []

for model_name, model_df in hp_df.groupby("model"):
    param_cols = [
        c for c in model_df.columns
        if c not in [
            "model",
            "fingerprint",
            "protocol",
            "fold",
            "inner_pr_auc",
            "test_pr_auc",
            "model_order",
            "fingerprint_order",
            "protocol_order",
        ]
    ]

    for protocol, protocol_df in model_df.groupby("protocol"):
        row = {
            "model": model_name,
            "protocol": protocol,
        }

        for col in param_cols:
            if protocol_df[col].notna().any():
                row[col] = unique_values_as_string(protocol_df[col])

        hp_set_summary_rows.append(row)

hp_set_summary = pd.DataFrame(hp_set_summary_rows)
hp_set_summary["model_order"] = hp_set_summary["model"].map(order_model)
hp_set_summary["protocol_order"] = hp_set_summary["protocol"].map(order_protocol)

hp_set_summary = hp_set_summary.sort_values(
    ["model_order", "protocol_order"]
).drop(columns=["model_order", "protocol_order"]).reset_index(drop=True)

hp_set_summary

,model,protocol,ccp_alpha,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,C,l1_ratio
0,Decision Tree,OOD holdout,"{0.001, 0.0001, 0.01}",{balanced},"{gini, entropy}","{20.0, 15.0, 7.0}","{sqrt, log2}","{5.0, 1.0, 2.0}","{20.0, 5.0, 2.0}",NaN,NaN
1,Decision Tree,Random shuffle,"{0.0, 0.0001}",{balanced},"{gini, entropy}","{20.0, 15.0}",{sqrt},"{5.0, 10.0, 2.0}","{2.0, 20.0, 10.0}",NaN,NaN
2,Logistic Regression,OOD holdout,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.1, 0.05, 1.0, 10.0, 5.0, 0.01}","{0.25, 0.0, 1.0}"
3,Logistic Regression,Random shuffle,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{1.0, 0.5, 0.1, 10.0}","{0.25, 0.0, 0.5, 1.0}"
4,Linear SVM,OOD holdout,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{100.0, 0.01, 0.001, 1.0, 25.0, 50.0}",NaN
5,Linear SVM,Random shuffle,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.25, 0.1, 25.0, 100.0, 10.0}",NaN


# Save processed tables for the plotting notebook

In [13]:
TASK = "hi"
DATASET = "drd2"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "results_ood_vs_random_shuffle"
    / TASK
    / DATASET
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Main protocol comparison tables
df_folds.to_csv(
    OUTPUT_DIR / "protocol_per_fold.csv",
    index=False,
)

summary_numeric.to_csv(
    OUTPUT_DIR / "protocol_summary_numeric.csv",
    index=False,
)

summary_table.to_csv(
    OUTPUT_DIR / "protocol_summary_display.csv",
    index=False,
)

delta_table.to_csv(
    OUTPUT_DIR / "protocol_delta.csv",
    index=False,
)

# Hyperparameter tables
hp_df.to_csv(
    OUTPUT_DIR / "hyperparameters_all.csv",
    index=False,
)

lr_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_lr.csv",
    index=False,
)

svm_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_svm.csv",
    index=False,
)

dt_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_dt.csv",
    index=False,
)

hp_set_summary.to_csv(
    OUTPUT_DIR / "hyperparameters_set_summary.csv",
    index=False,
)

print("Saved processed files in:")
print(OUTPUT_DIR)

print("\nFiles saved:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {file.name}")

Saved processed files in:
/home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/drd2

Files saved:
- complexity_all.csv
- complexity_dt.csv
- complexity_gap_analysis.csv
- complexity_lr.csv
- complexity_summary.csv
- complexity_svm.csv
- feature_importance_all.csv
- feature_importance_summary.csv
- feature_overlap_protocol.csv
- feature_stability_intra_protocol.csv
- feature_topk.csv
- hyperparameters_all.csv
- hyperparameters_dt.csv
- hyperparameters_lr.csv
- hyperparameters_set_summary.csv
- hyperparameters_svm.csv
- local_feature_contributions.csv
- local_molecule_candidates.csv
- protocol_delta.csv
- protocol_per_fold.csv
- protocol_summary_display.csv
- protocol_summary_numeric.csv
